# Jira API test

This notebook fetches Jira issues using Jira's enhanced JQL endpoint and builds a `tickets` list.
It parses `response["issues"]` and handles token-based pagination (`nextPageToken`).

In [ ]:
import requests

# Fill these in
BASE_URL = "https://innolab-reqeng.atlassian.net"
EMAIL = "robert.guenzler@studium.uni-hamburg.de"
API_TOKEN = ""

JQL = "project = ABC ORDER BY created DESC"

PROJECT = "SCRUM"

In [29]:
import requests
from requests.auth import HTTPBasicAuth

email = EMAIL
api_token = API_TOKEN
site = BASE_URL
project_key = "SCRUM"

r = requests.post(
    f"{site}/rest/api/3/search/jql",
    auth=HTTPBasicAuth(email, api_token),
    headers={
        "Accept": "application/json",
        "Content-Type": "application/json",
    },
    json={
        "jql": f"project = {project_key}",
        "maxResults": 10,
    },
)

print(r.status_code)
print(r.text)

200
{"issues":[{"id":"10036"},{"id":"10003"},{"id":"10002"},{"id":"10001"},{"id":"10000"}],"isLast":true}


In [44]:
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
import json

url = f"{BASE_URL}/rest/api/3/search/jql"

response = requests.post(
    url,
    auth=HTTPBasicAuth(EMAIL, API_TOKEN),
    headers={
        "Accept": "application/json",
        "Content-Type": "application/json",
    },
    json={
        "jql": f"project = {PROJECT} ORDER BY created DESC",
        "maxResults": 10,
        "fields": [
            "summary",
            "description",
            "status",
            "issuetype",
            "priority",
            "assignee",
            "reporter",
            "created",
            "updated"
        ]
    }
)

print(response.status_code)

data = response.json()

rows = []

for issue in data["issues"]:
    fields = issue["fields"]
    
    rows.append({
        "key": issue["key"],
        "summary": fields.get("summary"),
        "status": (fields.get("status") or {}).get("name"),
        "type": (fields.get("issuetype") or {}).get("name"),
        "priority": (fields.get("priority") or {}).get("name"),
        "assignee": (fields.get("assignee") or {}).get("displayName"),
        "reporter": (fields.get("reporter") or {}).get("displayName"),
        "created": fields.get("created"),
        "updated": fields.get("updated"),
    })

df = pd.DataFrame(rows)

print(df.head())

# for issue in data["issues"]:
#     print("KEY:", issue["key"])
#     print("SUMMARY:", issue["fields"].get("summary"))
#     print("STATUS:", issue["fields"].get("status", {}).get("name"))
#     print("TYPE:", issue["fields"].get("issuetype", {}).get("name"))
#     print("ASSIGNEE:", (issue["fields"].get("assignee") or {}).get("displayName"))
#     print("REPORTER:", (issue["fields"].get("reporter") or {}).get("displayName"))
#     print("CREATED:", issue["fields"].get("created"))
#     print("UPDATED:", issue["fields"].get("updated"))
#     print("DESCRIPTION:")
#     print(json.dumps(issue["fields"].get("description"), indent=2, ensure_ascii=False))
#     print("-" * 80)

200
       key           summary        status     type priority        assignee  \
0  SCRUM-5  Beispiel Story 1  Zu erledigen    Story   Medium             NaN   
1  SCRUM-4      Sub-Task 2.1  Zu erledigen  Subtask      NaN             NaN   
2  SCRUM-3         Aufgabe 3     In Arbeit     Task      NaN             NaN   
3  SCRUM-2         Aufgabe 2     In Arbeit    Story      NaN  Robert Günzler   
4  SCRUM-1         Aufgabe 1  Zu erledigen     Task      NaN             NaN   

         reporter                       created                       updated  
0  Robert Günzler  2026-03-31T13:52:09.833+0200  2026-03-31T13:52:23.207+0200  
1  Robert Günzler  2026-03-31T10:53:00.904+0200  2026-03-31T10:53:01.527+0200  
2  Robert Günzler  2026-03-31T10:52:59.273+0200  2026-03-31T10:53:00.400+0200  
3  Robert Günzler  2026-03-31T10:52:59.055+0200  2026-03-31T11:24:20.929+0200  
4  Robert Günzler  2026-03-31T10:52:58.338+0200  2026-03-31T10:53:01.956+0200  
